<a href="https://colab.research.google.com/github/chiyanglin-AStar/Python_3D_Lib/blob/main/Jax_tutorial_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation Steps

First, we need to install JAX and `jaxlib`. The installation command depends on whether you want CPU, GPU, or TPU support. Colab environments typically offer GPU or TPU runtimes, which can be selected from `Runtime -> Change runtime type`.

For GPU support (recommended in Colab if available):

## JAX Tutorial in Google Colab

JAX is a high-performance numerical computing library, designed for high-performance machine learning research. It combines **automatic differentiation (Autograd)** for differentiating native Python and NumPy functions, with **XLA (Accelerated Linear Algebra)** for compiling and running your NumPy programs on GPUs and TPUs. This allows JAX to achieve impressive performance and flexibility, especially for deep learning.

### Key Features of JAX:

1.  **`jax.grad`**: Automatic differentiation for scalar-valued functions.
2.  **`jax.jit`**: Just-in-time compilation for accelerating functions.
3.  **`jax.vmap`**: Automatic vectorization (batching) of functions.
4.  **`jax.pmap`**: Automatic parallelization across multiple devices.

## Numba 在 3D 應用中的功能

Numba 本身並沒有直接的「3D 功能」或圖形渲染能力。它是一個 JIT 編譯器，主要目的是加速 Python 的數值計算。然而，在許多 3D 應用中（例如 3D 模型處理、物理模擬、科學視覺化等），Numba 可以在底層的數值運算層面提供巨大的效能提升，尤其是當這些運算涉及到大量重複的數學操作時。換句話說，Numba 幫助您加速處理 3D 資料的程式碼，而不是直接處理 3D 渲染或互動。

以下是 Numba 的主要功能如何應用於 3D 相關任務：

1.  **核心數值運算加速 (`@jit`)**:
    *   對於處理 3D 向量、矩陣運算（例如旋轉、縮放、平移）、碰撞檢測、幾何演算法等，使用 `@jit(nopython=True)` 可以將 Python 迴圈和數學運算編譯成高效的機器碼，大幅提升速度。
    *   例如，計算 3D 點雲中的距離、找出最近鄰點，或對體積資料進行迭代處理。

2.  **元素級運算 (`@vectorize`)**:
    *   當你需要對 3D 陣列（如體積資料或點雲的座標陣列）中的每個元素執行相同的數學運算時，`@vectorize` 可以將標量函數轉換為高效的 NumPy ufunc，在 CPU 上實現並行化處理，避免 Python 迴圈的開銷。
    *   這對於對每個體素應用函數，或對每個頂點的座標進行調整非常有用。

3.  **CPU 平行化 (`prange`)**:
    *   對於可以在多個 CPU 核心上獨立執行的 3D 任務，例如處理大型網格的每個單元格，或獨立處理點雲中的每個點子集，`prange` 配合 `@jit(parallel=True)` 可以自動將迴圈分散到多個 CPU 核心，加速處理。

4.  **GPU 加速 (`cuda`) - 最適合 3D 相關任務**:
    *   這是 Numba 在 3D 領域中影響力最大的功能。許多 3D 運算本質上是高度並行的，非常適合 GPU。
    *   **自訂 CUDA 核心**: 您可以使用 `@cuda.jit` 編寫 CUDA 核心來直接在 GPU 上執行。
        *   **體積渲染 (Volume Rendering)**：加速光線追蹤 (ray casting) 或光線行進 (ray marching) 演算法。
        *   **點雲處理 (Point Cloud Processing)**：例如，對數百萬個點進行濾波、下採樣、特徵提取（如法線計算）。
        *   **物理模擬 (Physics Simulations)**：加速基於網格的流體模擬、粒子系統等。
        *   **3D 影像處理**：對醫學影像（如 CT、MRI 掃描）等 3D 數據進行濾波、分割、重建等。
        *   **幾何處理**：例如，Marching Cubes 演算法用於從體積數據生成表面網格。
    *   **CUDA 陣列與資料傳輸**: 有效地將 3D 數據（如 NumPy 陣列）從 CPU 記憶體傳輸到 GPU 記憶體 (`cuda.to_device`)，並在運算完成後傳回 (`copy_to_host`)，這是 GPU 效能優化的關鍵。
    *   **執行配置**: 設定適當的 `threads_per_block` 和 `blocks_per_grid`，甚至使用 `cuda.grid(3)` 處理 3D 網格資料，以充分利用 GPU 的並行處理能力。

總之，Numba 透過提供強大的數值計算加速能力，讓 Python 程式在處理 3D 數據時能夠達到接近 C/C++ 的效能，尤其是在結合 CUDA 進行 GPU 運算時。

### Verify Installation

After installation, it's good practice to verify that JAX is correctly installed and can detect your hardware accelerator (GPU or TPU if available).

### 1. `jax.grad`: 自動微分 (Automatic Differentiation)

`jax.grad` 是 JAX 最核心的功能之一，它允許我們輕鬆地計算 Python 函數的梯度。這在訓練神經網絡和其他優化問題中非常有用。

我們將定義一個簡單的函數，然後使用 `jax.grad` 來計算它的導數。

### 2. `jax.jit`: Just-In-Time 編譯

`jax.jit` 是一個函數轉換 (function transformation)，它會將您的 Python/NumPy 函數即時 (Just-In-Time) 編譯成 XLA (Accelerated Linear Algebra) 優化過的程式碼。這可以顯著提升函數的執行速度，尤其是在重複執行相同計算、且輸入形狀 (shape) 不變的情況下。當您在 GPU 或 TPU 上運行 JAX 時，`jax.jit` 的加速效果會更加明顯。

`jax.jit` 的主要優點：
-   **效能提升**：通過 XLA 編譯器，將多個 JAX 操作融合 (fuse) 成一個優化過的大型操作，減少了 Python 的解釋器開銷。
-   **設備加速**：充分利用 GPU 和 TPU 的並行計算能力。
-   **首次運行開銷**：第一次執行 `jitted` 函數時會有編譯開銷，之後的調用會非常快。

**使用注意事項**：
-   `jax.jit` 要求函數的輸入形狀通常保持不變。如果輸入形狀經常變化，每次變換都將重新編譯，可能導致性能下降。
-   函數內部應盡量避免 Python 的控制流（如 `if/else`, `for` 迴圈），除非這些控制流可以被 JAX 的控制流原語（如 `jax.lax.cond`, `jax.lax.while_loop`）靜態化或轉換。

我們將使用一個簡單的矩陣乘法範例來展示 `jax.jit` 的加速效果。

In [ ]:
# Install JAX with GPU support. If you are using a CPU runtime, you would install `jax[cpu]`.
# The exact version of jaxlib needs to match your CUDA version. Colab usually has recent CUDA.
# We'll try to install the latest version for CUDA 12.
!pip install --upgrade jax[cuda12_pip]
# If the above fails, you might need to find a specific jaxlib version for your CUDA:
# !pip install jaxlib==0.4.23+cuda12.cudnn89 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Or for CPU only (if you don't have a GPU/TPU runtime selected):
# !pip install --upgrade jax[cpu]

print("JAX and jaxlib installation initiated.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 70.9 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 455, in run
    installed = install_given_reqs(
                ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/req/__init__.py", line 70, in install_given_reqs
    requirement.insta

In [ ]:
import jax
import jax.numpy as jnp
import time

# Define a function without JIT
def matrix_multiply(x, y):
    return jnp.dot(x, y)

# Define the same function with JIT
@jax.jit
def jitted_matrix_multiply(x, y):
    return jnp.dot(x, y)

# Create large matrices
key = jax.random.PRNGKey(0)
size = 1000
matrix_a = jax.random.normal(key, (size, size))
matrix_b = jax.random.normal(key, (size, size))

print(f"Performing {size}x{size} matrix multiplication...")

# Measure execution time without JIT
start_time = time.time()
result_non_jit = matrix_multiply(matrix_a, matrix_b).block_until_ready() # .block_until_ready() ensures computation completes
end_time = time.time()
print(f"Time without JIT: {end_time - start_time:.4f} seconds")

# Measure execution time with JIT
# First run includes compilation time
start_time = time.time()
result_jit_first = jitted_matrix_multiply(matrix_a, matrix_b).block_until_ready()
end_time = time.time()
print(f"Time with JIT (first run, including compilation): {end_time - start_time:.4f} seconds")

# Subsequent runs are much faster as the function is already compiled
start_time = time.time()
result_jit_second = jitted_matrix_multiply(matrix_a, matrix_b).block_until_ready()
end_time = time.time()
print(f"Time with JIT (second run, compiled): {end_time - start_time:.4f} seconds")

# Verify results are the same
print(f"Results are approximately equal: {jnp.allclose(result_non_jit, result_jit_first)}")

ModuleNotFoundError: jax requires jaxlib to be installed. See https://github.com/jax-ml/jax#installation for installation instructions.